# 02 Cleaning

Use this notebook to clean the raw data, document your transformations, and export a processed file to `data/processed/`.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/raw/retail_store_sales.csv")

In [3]:
df.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  str    
 1   Customer ID       12575 non-null  str    
 2   Category          12575 non-null  str    
 3   Item              11362 non-null  str    
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  str    
 8   Location          12575 non-null  str    
 9   Transaction Date  12575 non-null  str    
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(1), str(7)
memory usage: 1.1+ MB


In [5]:
df.isnull().sum()

Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64

In [6]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df

,transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False
...,...,...,...,...,...,...,...,...,...,...,...
12570,TXN_9347481,CUST_18,Patisserie,Item_23_PAT,38.0,4.0,152.0,Credit Card,In-store,2023-09-03,NaN
12571,TXN_4009414,CUST_03,Beverages,Item_2_BEV,6.5,9.0,58.5,Cash,Online,2022-08-12,False
12572,TXN_5306010,CUST_11,Butchers,Item_7_BUT,14.0,10.0,140.0,Cash,Online,2024-08-24,NaN
12573,TXN_5167298,CUST_04,Furniture,Item_7_FUR,14.0,6.0,84.0,Cash,Online,2023-12-30,True


In [7]:
# Case 1: Price + Quantity → Total
mask = df["price_per_unit"].notnull() & df["quantity"].notnull() & df["total_spent"].isnull()
df.loc[mask, "total_spent"] = df.loc[mask, "price_per_unit"] * df.loc[mask, "quantity"]

# Case 2: Price + Total → Quantity
mask = df["price_per_unit"].notnull() & df["total_spent"].notnull() & df["quantity"].isnull()
df.loc[mask, "quantity"] = df.loc[mask, "total_spent"] / df.loc[mask, "price_per_unit"]

# Case 3: Quantity + Total → Price
mask = df["quantity"].notnull() & df["total_spent"].notnull() & df["price_per_unit"].isnull()
df.loc[mask, "price_per_unit"] = df.loc[mask, "total_spent"] / df.loc[mask, "quantity"]


# Case 4: Only Price exists → assume quantity = 1
mask = df["price_per_unit"].notnull() & df["quantity"].isnull() & df["total_spent"].isnull()
df.loc[mask, "quantity"] = 1
df.loc[mask, "total_spent"] = df.loc[mask, "price_per_unit"]

# Case 5: Only Total exists → assume quantity = 1
mask = df["total_spent"].notnull() & df["price_per_unit"].isnull() & df["quantity"].isnull()
df.loc[mask, "quantity"] = 1
df.loc[mask, "price_per_unit"] = df.loc[mask, "total_spent"]

# Case 6: Only Quantity exists → use median price
median_price = df["price_per_unit"].median()

mask = df["quantity"].notnull() & df["price_per_unit"].isnull() & df["total_spent"].isnull()
df.loc[mask, "price_per_unit"] = median_price
df.loc[mask, "total_spent"] = df.loc[mask, "quantity"] * df.loc[mask, "price_per_unit"]


df["total_spent"] = df["price_per_unit"] * df["quantity"]

# --- STEP 4: Clean formatting ---

df["quantity"] = df["quantity"].round(0).astype("Int64")
df["price_per_unit"] = df["price_per_unit"].round(2)
df["total_spent"] = df["total_spent"].round(2)

In [8]:
df["discount_applied"] = df["discount_applied"].fillna(False)

In [9]:
df["item"] = df["item"].fillna("Unknown_Item")

In [10]:
df["item"].nunique()

201

In [11]:
df["transaction_date"] = pd.to_datetime(df["transaction_date"])
df["discount_applied"] = df["discount_applied"].astype(bool)

In [12]:
df.drop_duplicates(inplace=True)

In [13]:
df.isnull().sum()

transaction_id      0
customer_id         0
category            0
item                0
price_per_unit      0
quantity            0
total_spent         0
payment_method      0
location            0
transaction_date    0
discount_applied    0
dtype: int64

In [14]:
df["month"] = df["transaction_date"].dt.month
df["day"] = df["transaction_date"].dt.day_name()
df["year"] = df["transaction_date"].dt.year

In [17]:
df.to_csv("../data/processed/clean_retail_data.csv", index=False)